# Exam Marks Prediction Using Machine Learning

A guided, reproducible exploration of the synthetic teaching dataset. Run cells from top to bottom. The data are fictional and the results do not establish real-world accuracy.

## 1. Load data and understand its structure
The identifier is excluded from modeling. `Final_Exam_Marks` is the regression target.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd()
if not (ROOT / 'dataset').exists(): ROOT = ROOT.parent
DATA_PATH = ROOT / 'dataset' / 'student_exam_data.csv'
data = pd.read_csv(DATA_PATH)
print('Shape:', data.shape)
display(data.head())
display(data.tail())
display(data.dtypes.to_frame('dtype'))
display(data.describe().T)
print('Missing values by column:\n', data.isna().sum())
print('Duplicate rows:', data.duplicated().sum())

## 2. Clean and prepare
Remove exact duplicates, turn infinite values into missing values, and drop rows without a target. Input medians are learned inside each model pipeline from training data only. We inspect rather than automatically discard unusual values.

In [ ]:
data = data.drop_duplicates().copy()
data = data.replace([np.inf, -np.inf], np.nan)
target = 'Final_Exam_Marks'
features = [c for c in data.columns if c not in ('Student_ID', target)]
data[target] = pd.to_numeric(data[target], errors='coerce')
data = data.dropna(subset=[target])
data[target] = data[target].clip(0, 100)
print('Cleaned shape:', data.shape)
print('Input features:', features)
print('Feature ranges:\n', data[features].agg(['min','max']).T)

## 3. Exploratory data analysis
Scatterplots show possible relationships; they do not establish causation. The heatmap summarizes pairwise linear correlation.

In [ ]:
sns.set_theme(style='whitegrid')
for feature in ['Study_Hours','Attendance','Previous_Exam_Marks','Internal_Marks','Sleep_Hours']:
    plt.figure(figsize=(6,4))
    sns.regplot(data=data, x=feature, y=target, scatter_kws={'alpha':0.45}, line_kws={'color':'red'})
    plt.title(f'{feature} vs Final Exam Marks'); plt.tight_layout(); plt.show()

plt.figure(figsize=(7,4)); sns.histplot(data[target], bins=20, kde=True); plt.title('Final exam mark distribution'); plt.show()
plt.figure(figsize=(11,8)); sns.heatmap(data[features+[target]].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0); plt.title('Correlation heatmap'); plt.tight_layout(); plt.show()

**Interpretation:** Use the plots to describe observed trends in your run. The generated target is intentionally related to several inputs, so relationships are expected. Correlation and visual association do not mean that changing one factor will cause a mark to change.
## 4. Feature selection and model comparison
All eleven meaningful input fields are retained; `Student_ID` is excluded. Correlation is a useful first look, but nonlinear models can use patterns that pairwise correlation does not capture. Scaling is used for Linear Regression, KNN, and SVR; trees do not need it. The same 80/20 split is used for all models.

In [ ]:
X, y = data[features], data[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
models = {
 'Linear Regression': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LinearRegression()),
 'Decision Tree': make_pipeline(SimpleImputer(strategy='median'), DecisionTreeRegressor(max_depth=6, random_state=42)),
 'Random Forest': make_pipeline(SimpleImputer(strategy='median'), RandomForestRegressor(n_estimators=250, min_samples_leaf=2, random_state=42, n_jobs=-1)),
 'K-Nearest Neighbors': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), KNeighborsRegressor(n_neighbors=7)),
 'Support Vector Regression': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), SVR(C=10, epsilon=2, kernel='rbf'))}
rows, fitted = [], {}
for name, model in models.items():
    model.fit(X_train, y_train); pred = np.clip(model.predict(X_test), 0, 100); fitted[name] = model
    rows.append({'Model':name, 'MAE':mean_absolute_error(y_test,pred), 'MSE':mean_squared_error(y_test,pred), 'RMSE':mean_squared_error(y_test,pred)**0.5, 'R2':r2_score(y_test,pred)})
results = pd.DataFrame(rows).sort_values('RMSE').reset_index(drop=True)
display(results.round(3))
best_name = results.loc[0,'Model']; best_model = fitted[best_name]
print('Selected by lowest held-out RMSE:', best_name)

MAE is average absolute miss in marks; MSE squares misses; RMSE converts that squared-error average back to mark units; R² compares the model with predicting the test-set average. Lower errors and a higher R² are generally preferred. These scores describe only this synthetic random split.

In [ ]:
best_pred = np.clip(best_model.predict(X_test), 0, 100)
fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].scatter(y_test,best_pred,alpha=.6); axes[0].plot([0,100],[0,100],'r--'); axes[0].set(xlabel='Actual marks',ylabel='Predicted marks',title=f'Actual vs predicted: {best_name}')
residuals = y_test - best_pred
axes[1].scatter(best_pred,residuals,alpha=.6); axes[1].axhline(0,color='red',linestyle='--'); axes[1].set(xlabel='Predicted marks',ylabel='Actual - predicted',title='Residual plot')
plt.tight_layout(); plt.show()
plt.figure(figsize=(9,4)); sns.barplot(data=results,x='RMSE',y='Model',color='steelblue'); plt.title('Held-out RMSE by model (lower is better)'); plt.show()

## 5. Tree feature importance
Tree importance indicates how the fitted model used features for splitting. It is model-specific and not evidence of causation.

In [ ]:
forest = fitted['Random Forest'].named_steps['randomforestregressor']
importance = pd.Series(forest.feature_importances_, index=features).sort_values()
importance.plot.barh(figsize=(8,5), title='Random Forest feature importance'); plt.xlabel('Importance'); plt.tight_layout(); plt.show()

## 6. Reproduce the deployable artifact
From a notebook started in the project directory, the following trains and saves the winning model, metrics JSON, and comparison CSV for the Streamlit interface. The app refits the selected configuration on all available rows after selection; the displayed comparison remains the held-out test result.

In [ ]:
import subprocess
subprocess.run([sys.executable, str(ROOT / 'train_model.py')], check=True)